In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report


c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
random_state = 42

In [3]:
df = pd.read_csv('../Milestone 3 EDA/ppp_cleaned.csv')
df

,ProcessingMethod,BorrowerState,LoanStatus,Term,InitialApprovalAmount,CurrentApprovalAmount,ServicingLenderState,RuralUrbanIndicator,HubzoneIndicator,LMIIndicator,...,MORTGAGE_INTEREST_PROCEED_pct,RENT_PROCEED_pct,REFINANCE_EIDL_PROCEED_pct,HEALTH_CARE_PROCEED_pct,DEBT_INTEREST_PROCEED_pct,PROCEED_Per_Job,Fraud,DateApproved_int,ForgivenessDate_int,LoanStatusDate_int
0,0.0,48.0,2.0,24,769358.78,769358.78,11.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,12409.01,0,1588291200,1605830400,1608249600
1,0.0,48.0,2.0,24,736927.79,736927.79,11.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,10094.90,0,1588291200,1628726400,1632787200
2,0.0,48.0,2.0,24,691355.00,691355.00,29.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,9218.07,0,1588291200,1612915200,1615939200
3,0.0,48.0,2.0,24,499871.00,499871.00,29.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,23803.38,0,1588291200,1631232000,1634342400
4,0.0,48.0,2.0,24,367437.00,367437.00,37.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,14697.48,0,1588291200,1617840000,1629158400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
968527,0.0,56.0,2.0,24,150000.00,150000.00,54.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,10000.00,0,1585872000,1607472000,1610496000
968528,0.0,56.0,2.0,24,150000.00,150000.00,31.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,3452.38,0,1586822400,1604361600,1607385600
968529,1.0,56.0,2.0,60,150000.00,150000.00,54.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,29999.40,0,1613088000,1629158400,1631664000
968530,0.0,56.0,2.0,60,150000.00,150000.00,18.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,21428.57,0,1586908800,1645574400,1646697600


## Baseline

In [4]:
X = df.drop(columns='Fraud')
y = df['Fraud']

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2, 
                                                    stratify=y,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

### Default settings on RF

In [6]:
# Train using optimal parameters
model = RandomForestClassifier(
    random_state=random_state,
    n_jobs=-1,
)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [7]:
y_pred = model.predict(X_test)

In [8]:
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    193688
           1       0.00      0.00      0.00        19

    accuracy                           1.00    193707
   macro avg       0.50      0.50      0.50    193707
weighted avg       1.00      1.00      1.00    193707



c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

In [9]:
baseline_model_report_df = classification_report(y_test, y_pred, output_dict=True)

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

### Class Weight Balanced

In [10]:
# Train using optimal parameters
model = RandomForestClassifier(
    class_weight='balanced',
    random_state=random_state,
    n_jobs=-1
)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [11]:
y_pred = model.predict(X_test)

In [12]:
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    193688
           1       0.00      0.00      0.00        19

    accuracy                           1.00    193707
   macro avg       0.50      0.50      0.50    193707
weighted avg       1.00      1.00      1.00    193707



c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

In [13]:
baseline_balanced_model_report_df = classification_report(y_test, y_pred, output_dict=True)

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

## Undersampling

In [14]:
fraud = df[df['Fraud'] == 1]
non_fraud = df[df['Fraud'] == 0]
len(fraud), len(non_fraud)

(95, 968437)

In [15]:
# Undersample majority class to a 1:10 ratio
non_fraud_downsampled = non_fraud.sample(n=len(fraud) * 10, random_state=random_state)
non_fraud_downsampled

,ProcessingMethod,BorrowerState,LoanStatus,Term,InitialApprovalAmount,CurrentApprovalAmount,ServicingLenderState,RuralUrbanIndicator,HubzoneIndicator,LMIIndicator,...,MORTGAGE_INTEREST_PROCEED_pct,RENT_PROCEED_pct,REFINANCE_EIDL_PROCEED_pct,HEALTH_CARE_PROCEED_pct,DEBT_INTEREST_PROCEED_pct,PROCEED_Per_Job,Fraud,DateApproved_int,ForgivenessDate_int,LoanStatusDate_int
495859,1.0,25.0,2.0,60,175700.0,175700.0,52.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,4748.57,0,1617062400,1638835200,1641427200
751210,0.0,41.0,2.0,24,379200.0,379200.0,40.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,9248.78,0,1586390400,1611619200,1613433600
302413,0.0,14.0,2.0,24,238300.0,238300.0,14.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,8510.71,0,1586217600,1605571200,1615334400
199402,0.0,8.0,2.0,24,742528.0,742528.0,37.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,19540.21,0,1586649600,1640044800,1642809600
52948,1.0,5.0,2.0,60,882672.0,882672.0,37.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,18389.00,0,1615593600,1640131200,1642809600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
294662,0.0,13.0,2.0,24,298700.0,298700.0,13.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,21335.71,0,1586908800,1630972800,1635379200
204016,0.0,9.0,2.0,24,375100.0,375100.0,42.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,10717.14,0,1588032000,1623888000,1626912000
387773,0.0,20.0,2.0,24,603200.0,603200.0,20.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,11827.45,0,1586044800,1627948800,1631577600
378063,0.0,19.0,2.0,24,515000.0,515000.0,37.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,10510.20,0,1586995200,1614902400,1618876800


In [16]:
# Combine and shuffle
df_reduced = pd.concat([fraud, non_fraud_downsampled]).sample(frac=1).reset_index(drop=True)
df_reduced

,ProcessingMethod,BorrowerState,LoanStatus,Term,InitialApprovalAmount,CurrentApprovalAmount,ServicingLenderState,RuralUrbanIndicator,HubzoneIndicator,LMIIndicator,...,MORTGAGE_INTEREST_PROCEED_pct,RENT_PROCEED_pct,REFINANCE_EIDL_PROCEED_pct,HEALTH_CARE_PROCEED_pct,DEBT_INTEREST_PROCEED_pct,PROCEED_Per_Job,Fraud,DateApproved_int,ForgivenessDate_int,LoanStatusDate_int
0,0.0,11.0,2.0,24,4840457.50,4840457.50,29.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,11893.02,0,1586476800,1630281600,1632441600
1,1.0,23.0,2.0,60,1596546.00,1596546.00,32.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,14128.70,0,1611619200,1638748800,1642723200
2,0.0,20.0,2.0,24,186700.00,186700.00,27.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,8117.39,0,1586908800,1620259200,1622764800
3,0.0,47.0,2.0,24,297500.00,297500.00,46.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,13522.73,0,1587945600,1628467200,1636761600
4,1.0,5.0,1.0,60,161091.24,161091.00,5.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,8054.45,0,1611878400,-2208988800,1611878400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1040,0.0,16.0,2.0,24,191000.00,191000.00,16.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,9550.00,0,1586563200,1629936000,1631577600
1041,1.0,10.0,2.0,60,2000000.00,2000000.00,11.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,10152.28,0,1616803200,1647388800,1647907200
1042,1.0,56.0,2.0,60,780000.00,780000.00,54.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,21081.03,0,1610755200,1638921600,1641600000
1043,1.0,45.0,2.0,60,285700.00,285700.00,44.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,12986.36,0,1613520000,1635897600,1639008000


In [17]:
X = df_reduced.drop(columns='Fraud')
y = df_reduced['Fraud']

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2, 
                                                    stratify=y,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

In [19]:
cv = StratifiedKFold(n_splits=5)

### Recall as metric

In [20]:
# Parameter tuning
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 2, 32, log=True)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    
    model = RandomForestClassifier(
        class_weight='balanced',
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    return scores.mean()


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

top_5_recall_df = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_recall_df)

[I 2026-04-14 20:46:51,815] A new study created in memory with name: no-name-73336011-1e3d-407c-b473-720802d006d4
[I 2026-04-14 20:46:53,307] Trial 0 finished with value: 0.7891666666666667 and parameters: {'n_estimators': 105, 'max_depth': 7, 'min_samples_split': 3}. Best is trial 0 with value: 0.7891666666666667.
[I 2026-04-14 20:46:54,385] Trial 1 finished with value: 0.8025 and parameters: {'n_estimators': 48, 'max_depth': 20, 'min_samples_split': 9}. Best is trial 1 with value: 0.8025.
[I 2026-04-14 20:46:55,467] Trial 2 finished with value: 0.8158333333333333 and parameters: {'n_estimators': 44, 'max_depth': 5, 'min_samples_split': 7}. Best is trial 2 with value: 0.8158333333333333.
[I 2026-04-14 20:46:56,510] Trial 3 finished with value: 0.7891666666666667 and parameters: {'n_estimators': 26, 'max_depth': 6, 'min_samples_split': 4}. Best is trial 2 with value: 0.8158333333333333.
[I 2026-04-14 20:46:56,696] Trial 4 finished with value: 0.8158333333333335 and parameters: {'n_esti

,number,value,datetime_start,datetime_complete,duration,params_max_depth,params_min_samples_split,params_n_estimators,state
32,32,0.920833,2026-04-14 20:46:59.460530,2026-04-14 20:46:59.554873,0 days 00:00:00.094343,2,5,104,COMPLETE
12,12,0.920833,2026-04-14 20:46:57.506351,2026-04-14 20:46:57.583727,0 days 00:00:00.077376,2,10,83,COMPLETE
16,16,0.920833,2026-04-14 20:46:57.895209,2026-04-14 20:46:57.987623,0 days 00:00:00.092414,2,5,102,COMPLETE
13,13,0.920833,2026-04-14 20:46:57.583727,2026-04-14 20:46:57.679344,0 days 00:00:00.095617,2,9,84,COMPLETE
28,28,0.920833,2026-04-14 20:46:59.135038,2026-04-14 20:46:59.227281,0 days 00:00:00.092243,2,6,94,COMPLETE


In [21]:
# Train using optimal parameters
model_1 = RandomForestClassifier(
    class_weight='balanced',
    n_estimators=top_5_recall_df.head(1)['params_n_estimators'].item(),
    max_depth=top_5_recall_df.head(1)['params_max_depth'].item(),
    min_samples_split=top_5_recall_df.head(1)['params_min_samples_split'].item(),
    random_state=random_state
)
model_1.fit(X_train, y_train)

,n_estimators,104
,criterion,'gini'
,max_depth,2
,min_samples_split,5
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [22]:
y_pred = model_1.predict(X_test)

In [23]:
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.99      0.95      0.97       190
           1       0.64      0.95      0.77        19

    accuracy                           0.95       209
   macro avg       0.82      0.95      0.87       209
weighted avg       0.96      0.95      0.95       209



In [24]:
recall_model_report_df = classification_report(y_test, y_pred, output_dict=True)

## F1 as metric

In [25]:
# Parameter tuning
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 2, 32, log=True)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    
    model = RandomForestClassifier(
        class_weight='balanced',
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train, 
        y=y_train, 
        cv=cv,
        scoring='f1',
        n_jobs=-1,
    )
    return scores.mean()


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

top_5_f1_df = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_f1_df)

[I 2026-04-14 20:47:06,523] A new study created in memory with name: no-name-e541dcc8-166e-40ed-a4f2-736ee9852759
[I 2026-04-14 20:47:06,597] Trial 0 finished with value: 0.6676210415809221 and parameters: {'n_estimators': 77, 'max_depth': 3, 'min_samples_split': 4}. Best is trial 0 with value: 0.6676210415809221.
[I 2026-04-14 20:47:06,737] Trial 1 finished with value: 0.7644932844932846 and parameters: {'n_estimators': 132, 'max_depth': 5, 'min_samples_split': 5}. Best is trial 1 with value: 0.7644932844932846.
[I 2026-04-14 20:47:06,815] Trial 2 finished with value: 0.7631317701905938 and parameters: {'n_estimators': 61, 'max_depth': 17, 'min_samples_split': 9}. Best is trial 1 with value: 0.7644932844932846.
[I 2026-04-14 20:47:06,923] Trial 3 finished with value: 0.7786243386243386 and parameters: {'n_estimators': 110, 'max_depth': 5, 'min_samples_split': 7}. Best is trial 3 with value: 0.7786243386243386.
[I 2026-04-14 20:47:07,015] Trial 4 finished with value: 0.661411916854246 

,number,value,datetime_start,datetime_complete,duration,params_max_depth,params_min_samples_split,params_n_estimators,state
32,32,0.792935,2026-04-14 20:47:11.357601,2026-04-14 20:47:11.557995,0 days 00:00:00.200394,21,4,186,COMPLETE
24,24,0.792935,2026-04-14 20:47:09.879923,2026-04-14 20:47:10.081522,0 days 00:00:00.201599,30,4,185,COMPLETE
25,25,0.792935,2026-04-14 20:47:10.081522,2026-04-14 20:47:10.268852,0 days 00:00:00.187330,31,4,191,COMPLETE
29,29,0.792935,2026-04-14 20:47:10.843338,2026-04-14 20:47:11.029315,0 days 00:00:00.185977,31,4,184,COMPLETE
22,22,0.792935,2026-04-14 20:47:09.538279,2026-04-14 20:47:09.725117,0 days 00:00:00.186838,15,4,185,COMPLETE


In [26]:
# Train using optimal parameters
model_2 = RandomForestClassifier(
    class_weight='balanced',
    n_estimators=top_5_f1_df.head(1)['params_n_estimators'].item(),
    max_depth=top_5_f1_df.head(1)['params_max_depth'].item(),
    min_samples_split=top_5_f1_df.head(1)['params_min_samples_split'].item(),
    random_state=random_state
)
model_2.fit(X_train, y_train)

,n_estimators,186
,criterion,'gini'
,max_depth,21
,min_samples_split,4
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [27]:
y_pred = model_2.predict(X_test)

In [28]:
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.98      0.99      0.99       190
           1       0.89      0.84      0.86        19

    accuracy                           0.98       209
   macro avg       0.94      0.92      0.93       209
weighted avg       0.98      0.98      0.98       209



In [29]:
f1_model_report_df = classification_report(y_test, y_pred, output_dict=True)

### Analyze reports

In [34]:
# 2. Convert and transpose
recall_report_df = pd.DataFrame(recall_model_report_df).transpose()
f1_report_df = pd.DataFrame(f1_model_report_df).transpose()

# 3. Combine multiple reports (e.g., adding a 'Model' column for identification)
combined_df = pd.concat([recall_report_df, f1_report_df], keys=['PPP RF optimized for Recall', 'PPP RF optimized for F1'])

In [35]:
combined_df

precision    recall  f1-score  \
PPP RF optimized for Recall 0              0.994475  0.947368  0.970350   
                            1              0.642857  0.947368  0.765957   
                            accuracy       0.947368  0.947368  0.947368   
                            macro avg      0.818666  0.947368  0.868154   
                            weighted avg   0.962510  0.947368  0.951769   
PPP RF optimized for F1     0              0.984293  0.989474  0.986877   
                            1              0.888889  0.842105  0.864865   
                            accuracy       0.976077  0.976077  0.976077   
                            macro avg      0.936591  0.915789  0.925871   
                            weighted avg   0.975620  0.976077  0.975785   

                                             support  
PPP RF optimized for Recall 0             190.000000  
                            1              19.000000  
                            accuracy        0.947368  
                            macro avg     209.000000  
                            weighted avg  209.000000  
PPP RF optimized for F1     0             190.000000  
                            1              19.000000  
                            accuracy        0.976077  
                            macro avg     209.000000  
                            weighted avg  209.000000

In [32]:
# Recall model
importances = model_1.feature_importances_
feature_names = df.drop(columns='Fraud').columns
feature_imp_recall_df = pd.DataFrame({'Feature': feature_names, 'Gini Importance': importances}).sort_values(
    'Gini Importance', ascending=False)
feature_imp_recall_df

,Feature,Gini Importance
24,ForgivenessAmount,0.204918
39,ForgivenessDate_int,0.165690
27,ForgivenPercentage,0.137547
2,LoanStatus,0.116737
26,NotForgivenAmount,0.108818
40,LoanStatusDate_int,0.100746
38,DateApproved_int,0.032356
3,Term,0.020419
37,PROCEED_Per_Job,0.016742
0,ProcessingMethod,0.015200


In [33]:
# Recall model
importances = model_2.feature_importances_
feature_names = df.drop(columns='Fraud').columns
feature_imp_f1_df = pd.DataFrame({'Feature': feature_names, 'Gini Importance': importances}).sort_values(
    'Gini Importance', ascending=False)
feature_imp_f1_df

,Feature,Gini Importance
39,ForgivenessDate_int,1.592659e-01
24,ForgivenessAmount,1.462004e-01
27,ForgivenPercentage,1.124539e-01
26,NotForgivenAmount,1.000095e-01
40,LoanStatusDate_int,9.282154e-02
2,LoanStatus,7.414662e-02
38,DateApproved_int,4.951110e-02
37,PROCEED_Per_Job,2.814497e-02
3,Term,2.322377e-02
5,CurrentApprovalAmount,2.206470e-02
